# 2. Búsqueda de la mejor arquitectura MLP

Evaluamos, para cada combinación de lag y horizonte, las 584 arquitecturas del espacio de búsqueda (1 a 3 capas ocultas con 32, 64, 128 o 256 neuronas y todas las máscaras de Dropout(0.5)) con Group K-Fold como estrategia principal. Este notebook no entrena: lee los resultados guardados por `src/train_t2.py`.

Regla de selección: entre los candidatos con `gap_ratio < 0.15` elegimos el de menor `best_val_rmse`. Los RMSE de la búsqueda están en unidades estandarizadas de log(precio) y promediados sobre los 5 folds; el `holdout_rmse` está en USD y solo se reporta como evidencia final.

In [ ]:
# Rutas: en Colab se usa la carpeta de Drive; en el repositorio, la raíz del proyecto.
import os, glob, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DRIVE = Path("/content/drive/MyDrive/tarea2_series")
if DRIVE.exists():
    BASE, FIGS = DRIVE, DRIVE / "figs"
else:
    BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    FIGS = BASE / "notebooks" / "figs"
RESULTS = BASE / os.environ.get("T2_RESULTS_DIR", "results")
FIGS.mkdir(parents=True, exist_ok=True)

LAGS, HS, ESTRATEGIAS = [7, 14, 21, 28], [7, 14, 21, 28], ["gk", "fc", "kf"]
NOMBRE = {"gk": "Group K-Fold", "fc": "Forward Chaining", "kf": "K-Fold"}
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # orden categorico fijo
COL_EST = dict(zip(ESTRATEGIAS, COL))
COL_LAG = dict(zip(LAGS, COL))
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
                     "axes.prop_cycle": plt.cycler(color=COL), "lines.linewidth": 1.6,
                     "legend.frameon": False, "axes.titlesize": 11, "axes.labelsize": 10})

df = pd.read_csv(BASE / "data" / "btc_1d.csv", parse_dates=["date"])
y = df["close"].to_numpy(float)
DATES = df["date"].to_numpy()
N_HOLD = max(int(len(y) * 0.10), 56)
N_CV = len(y) - N_HOLD


def guardar(fig, nombre):
    fig.savefig(FIGS / f"{nombre}.png", bbox_inches="tight", dpi=150)

In [ ]:
def cargar_registros():
    R = {}
    for f in sorted(glob.glob(str(RESULTS / "raw" / "*.jsonl"))):
        for line in open(f):
            if line.strip():
                r = json.loads(line); R[r["id"]] = r
    return list(R.values())

REG = cargar_registros()
print(f"{len(REG)} registros en {RESULTS / 'raw'}")
print(pd.Series([r["phase"] for r in REG]).value_counts().to_string())

## 2.1 Tabla completa de la búsqueda

In [ ]:
def tabla_busqueda(reg):
    filas = []
    for r in reg:
        if r["phase"] != "search":
            continue
        filas.append(dict(lag=r["L"], H=r["H"], arquitectura=str(tuple(r["arch"])),
                          dropout_mask=str(tuple(r["mask"])), n_capas=len(r["arch"]),
                          ancho_medio=np.mean(r["arch"]), n_dropout=int(sum(r["mask"])),
                          best_val_rmse=r["best_val_rmse"], last_train_rmse=r["last_train_rmse"],
                          gap=r["gap"], gap_ratio=r["gap_ratio"], best_epoch=r["best_epoch"],
                          truncado=r["truncated"], segundos=r["secs"]))
    return pd.DataFrame(filas)

S = tabla_busqueda(REG)
for (L, H), g in S.groupby(["lag", "H"]):
    g.sort_values("best_val_rmse").to_csv(RESULTS / f"architecture_search_lag{L}_h{H}.csv", index=False)
cobertura = S.pivot_table(index="lag", columns="H", values="best_val_rmse", aggfunc="count").fillna(0).astype(int)
print("Candidatos evaluados por (lag, H) (de 584):")
cobertura

## 2.2 Mejor arquitectura por lag y horizonte

In [ ]:
def seleccionar(g):
    g = g[~g.truncado]
    bal = g[g.gap_ratio < 0.15]
    return (bal if len(bal) else g).sort_values("best_val_rmse"), len(bal) > 0

filas = []
for (L, H), g in S.groupby(["lag", "H"]):
    orden, hay_bal = seleccionar(g)
    b = orden.iloc[0]
    filas.append(dict(lag=L, H=H, arquitectura=b.arquitectura, dropout_mask=b.dropout_mask,
                      best_val_rmse=b.best_val_rmse, gap_ratio=b.gap_ratio,
                      candidatos_balanceados=int((g.gap_ratio < 0.15).sum()), evaluados=len(g),
                      cumple_filtro=hay_bal))
MEJOR = pd.DataFrame(filas)
MEJOR.to_csv(RESULTS / "mejor_arquitectura_por_config.csv", index=False)
MEJOR.round(4)

## 2.3 Top-5 por configuración con RMSE en el hold-out

In [ ]:
HO = {(r["L"], r["H"], str(tuple(r["arch"])), str(tuple(r["mask"]))): r["holdout_rmse"]
      for r in REG if r["phase"] == "top5"}
top5 = []
for (L, H), g in S.groupby(["lag", "H"]):
    orden, _ = seleccionar(g)
    t = orden.head(5)[["lag", "H", "arquitectura", "dropout_mask", "best_val_rmse", "gap_ratio"]].copy()
    t.insert(2, "rank", range(1, len(t) + 1))
    t["holdout_rmse_usd"] = [HO.get((L, H, a, m)) for a, m in zip(t.arquitectura, t.dropout_mask)]
    top5.append(t)
TOP5 = pd.concat(top5)
TOP5.to_csv(RESULTS / "top5_por_config.csv", index=False)
TOP5[(TOP5.lag == 14) & (TOP5.H == 14)].round(4)

## 2.4 Curvas de aprendizaje del Top-3 (lag 14, H 14)

In [ ]:
CURV = {(r["L"], r["H"], str(tuple(r["arch"])), str(tuple(r["mask"]))): r
        for r in REG if r["phase"] == "search"}

def curvas_top3(L, H, ax):
    for k, (_, f) in enumerate(TOP5[(TOP5.lag == L) & (TOP5.H == H)].head(3).iterrows()):
        r = CURV[(L, H, f.arquitectura, f.dropout_mask)]
        ep = np.arange(1, len(r["train_curve"]) + 1)
        ax.plot(ep, r["train_curve"], color=COL[k], ls="--", lw=1.2)
        ax.plot(ep, r["val_curve"], color=COL[k], lw=1.6, label=f"#{k+1} {f.arquitectura} {f.dropout_mask}")
    ax.set_title(f"lag {L}, H {H}"); ax.set_xlabel("Época"); ax.set_ylabel("RMSE (estandarizado)")

fig, ax = plt.subplots(figsize=(8, 4))
curvas_top3(14, 14, ax)
ax.legend(fontsize=8, title="línea continua: validación · discontinua: train", title_fontsize=8)
guardar(fig, "busqueda_top3_L14_H14"); plt.show()

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(15, 12), sharex=True)
for i, L in enumerate(LAGS):
    for j, H in enumerate(HS):
        if len(TOP5[(TOP5.lag == L) & (TOP5.H == H)]):
            curvas_top3(L, H, axs[i, j])
        axs[i, j].set_xlabel(""); axs[i, j].set_ylabel("")
fig.suptitle("Top-3 por configuración (continua: validación; discontinua: train)", y=1.0)
fig.tight_layout(); guardar(fig, "busqueda_top3_todas"); plt.show()

## 2.5 Efecto de profundidad, ancho y dropout

In [ ]:
efectos = {}
for var, nombre in (("n_capas", "Capas ocultas"), ("ancho_medio", "Neuronas por capa (media)"),
                    ("n_dropout", "Capas con dropout")):
    efectos[var] = S.groupby(var).agg(mediana_val_rmse=("best_val_rmse", "median"),
                                      mediana_gap_ratio=("gap_ratio", "median"),
                                      frac_balanceados=("gap_ratio", lambda s: float((s < 0.15).mean())),
                                      n=("best_val_rmse", "size"))
    print(f"\n{nombre}\n", efectos[var].round(4).to_string())

fig, axs = plt.subplots(2, 3, figsize=(14, 7))
for j, var in enumerate(("n_capas", "n_dropout", "ancho_medio")):
    cats = sorted(S[var].unique()) if var != "ancho_medio" else None
    if var == "ancho_medio":
        S["_ancho"] = pd.cut(S.ancho_medio, [0, 48, 96, 160, 256], labels=["≤48", "49–96", "97–160", ">160"])
        cats, v = list(S["_ancho"].cat.categories), "_ancho"
    else:
        v = var
    for i, met in enumerate(("best_val_rmse", "gap_ratio")):
        datos = [S.loc[S[v] == c, met].dropna() for c in cats]
        axs[i, j].boxplot(datos, showfliers=False, medianprops=dict(color=COL[1]), boxprops=dict(color=COL[0]))
        axs[i, j].set_xticks(range(1, len(cats) + 1)); axs[i, j].set_xticklabels([str(c) for c in cats])
        axs[i, j].set_xlabel({"n_capas": "Capas ocultas", "n_dropout": "Capas con dropout",
                              "ancho_medio": "Neuronas por capa (media)"}[var])
        axs[i, j].set_ylabel(met)
axs[1, 0].axhline(0.15, color="0.4", ls=":", lw=1)
fig.suptitle("Distribución sobre todas las configuraciones evaluadas"); fig.tight_layout()
guardar(fig, "busqueda_efectos"); plt.show()

## 2.6 ¿Cambia la arquitectura óptima con el horizonte?

In [ ]:
pd.pivot_table(MEJOR.assign(sel=MEJOR.arquitectura + " " + MEJOR.dropout_mask),
               index="lag", columns="H", values="sel", aggfunc="first")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for L in LAGS:
    g = MEJOR[MEJOR.lag == L].sort_values("H")
    ax.plot(g.H, g.best_val_rmse, marker="o", ms=6, color=COL_LAG[L], label=f"lag {L}")
ax.set_xticks(HS); ax.set_xlabel("Horizonte H (días)"); ax.set_ylabel("best_val_rmse (estandarizado)")
ax.set_title("Error de validación de la arquitectura seleccionada"); ax.legend()
guardar(fig, "busqueda_mejor_vs_H"); plt.show()